# Standalone RAG Implementation for Google Colab
This notebook demonstrates Retrieval-Augmented Generation (RAG) using the `beir/webis-touche2020/v2` dataset. It is completely self-contained and runs directly in Google Colab.

### 1. Install Dependencies

In [ ]:
!pip install transformers torch sentence-transformers ir_datasets faiss-cpu tqdm

^C


   ---------------------------------------- 0.0/16.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.2 MB ? eta -:--:--
    --------------------------------------- 0.3/16.2 MB ? eta -:--:--
    ----------------

### 2. Load the Dataset
We use `ir_datasets` to load the `beir/webis-touche2020/v2` dataset. To make this experiment run quickly, we will only index the first 10,000 documents.

In [ ]:
import ir_datasets
from tqdm.auto import tqdm

# Load the dataset
dataset = ir_datasets.load("beir/webis-touche2020/v2")
documents = []
doc_ids = []

for i, doc in enumerate(tqdm(dataset.docs_iter(), total=max_docs)):
    documents.append(doc.text)
    doc_ids.append(doc.doc_id)

print(f"Loaded {len(documents)} documents.")

  0%|          | 0/10000 [00:00<?, ?it/s]

[INFO] Opening /root/.ir_datasets/beir/webis-touche2020/v2/docs.pklz4/bin with direct file access


Loaded 382545 documents.


### 3. Build the Vector Index
We use `sentence-transformers` to generate embeddings and `faiss` to search through them.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print("Loading embedding model...")
import os
local_embed_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'models', 'minilm-local'))
if os.path.exists(local_embed_path):
    print(f'Loading local embedding model from {local_embed_path}...')
    embedding_model = SentenceTransformer(local_embed_path)
else:
    print('Loading embedding model from HuggingFace...')
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Encoding documents (this might take a minute)...")
doc_embeddings = embedding_model.encode(documents, batch_size=64, show_progress_bar=True)

print("Building FAISS index...")
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings).astype('float32'))

print("Index built successfully!")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding documents (this might take a minute)...


Batches:   0%|          | 0/5978 [00:00<?, ?it/s]

Building FAISS index...
Index built successfully!


### 4. Initialize the Language Model
We will use `google/flan-t5-base` as our LLM for generating answers based on retrieved context.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

import os
local_llm_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'models', 'flan-t5-local'))
model_name = local_llm_path if os.path.exists(local_llm_path) else 'google/flan-t5-base'
print(f"Loading LLM: {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
print("LLM Loaded successfully!")

Loading LLM: google/flan-t5-base...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


LLM Loaded successfully!


### 5. Define the RAG Pipeline
Here we combine the Retrieval step (FAISS) and the Generation step (Flan-T5).

In [ ]:
def experiment_rag(query: str, top_k: int = 3, max_new_tokens: int = 100):
    # A. Retrieve top-k documents
    print(f"Retrieving top {top_k} documents for query: '{query}'")
    query_embedding = embedding_model.encode([query]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)

    # B. Fetch full context
    context_texts = []
    retrieved_doc_ids = []
    for idx in indices[0]:
        if idx != -1:
            context_texts.append(documents[idx]) # First 500 chars
            retrieved_doc_ids.append(doc_ids[idx])

    context = "\n---\n".join(context_texts)

    # C. Build the Prompt
    prompt = (
        "Answer the question using ONLY the provided context.\n\n"
        f"Context:\n{context}\n\n"
        f"Question:\n{query}\n\n"
        "Answer:\n"
    )
    print("\n--- PROMPT ---")
    print(prompt)
    print("--------------\n")

    # D. Generate Answer
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = llm_model.generate(**inputs, max_new_tokens=max_new_tokens)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer, retrieved_doc_ids

### 6. Test the Pipeline

In [ ]:
# Example from Webis-Touche2020
query = "Should any vaccines be required for children?"
answer, doc_results = experiment_rag(query, top_k=10, max_new_tokens=50)

print("\n========== FINAL RAG ANSWER ==========")
print(answer)
print("======================================\n")
print("Retrieved Document IDs:", doc_results)

: 